<a href="https://colab.research.google.com/github/Wings-hub/Gait-Pattern-Nano-33-BLE-Sense/blob/main/Implementation/Gait_pattern_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Human Activity Pattern Recognition**

# Introduction :
In this project, we develop a machine learning model tailored for recognizing six distinct human activities: sitting, jumping, ascending stairs, descending stairs, walking, and standing. Utilizing Keras, a user-friendly deep learning framework, the model leverages the power of 1D Convolutional Neural Networks (CNNs) specifically designed for time-series data. Upon training, the model is converted to TensorFlow Lite, making it compatible for deployment on lightweight devices, such as the Arduino Nano 33 BLE. This approach ensures real-time, efficient, and accurate activity recognition, paving the way for innovative applications in health, fitness, and assistive technologies.


# Structure :

*   Installation of Packages
*   Mounting Drive and Setting up Kaggle
*   Setup Kaggle directory and permissions.
*    Download and Extract Dataset

*   Import Libraries

*    Define Dataset Exploration and Preparation Functions

*    Load and Process Dataset

*    Create and normalize the dataset.

*    Define window size and stride.
*    Data Splitting

*    Define and Compile the Model


*    Training the Model

*    Evaluate the Model

*     Convert to TensorFlow Lite


*     Conversion to .h and .cpp files




# Installation of Packages

In [ ]:
pip install kaggle

In [ ]:
pip install pandas numpy tensorflow


# Mounting Drive and Setting up Kaggle



  Mount Google Drive for accessing data and setup Kaggle directory and permissions. Downloading the Kaggle.jason to The drive by creating a new token on Kaggle.



In [ ]:
# Upload data files for processing
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p ~/.kaggle


In [ ]:
! cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json


# Download and Extract Dataset

 Download the MotionSense dataset from Kaggle and extract it to the designated directory.Here we used the MotionSense Dataset : Smartphone Sensor Data - HAR built from the recordings of 24 participants performing daily activities like downstairs, upstairs, walking, jogging, sitting, and standing, using an iPhone 6s placed in their front pocket. The phone's accelerometer and gyroscope, accessed via SensingKit from the Core Motion framework, captured the data.

 More info : https://www.kaggle.com/datasets/malekzadeh/motionsense-dataset

In [ ]:
!kaggle datasets download -d malekzadeh/motionsense-dataset


 92% 66.0M/72.0M [00:00<00:00, 61.0MB/s]
100% 72.0M/72.0M [00:00<00:00, 76.6MB/s]


In [ ]:
!unzip /content/motionsense-dataset.zip -d /content/


Archive:  /content/motionsense-dataset.zip
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_1.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_10.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_11.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_12.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_13.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_14.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_15.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_16.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_17.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_18.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1/sub_19.csv  
  inflating: /content/A_DeviceMotion_data/A_DeviceMotion_data/dws_1

In [ ]:
!ls /content/



activity_recognition_model_data.cpp  drive
activity_recognition_model_data.h    motionsense-dataset.zip
A_DeviceMotion_data		     sample_data
data_subjects_info.csv		     sine_model_quantized.cc


# Import Libraries

Import necessary libraries like os, numpy, pandas, TensorFlow, etc.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
import gc


# Define Dataset Exploration and Preparation Functions
Several functions are defined to retrieve, preprocess and explore the dataset.

In [ ]:
def get_ds_infos():
    dss = pd.read_csv("data_subjects_info.csv")
    print("[INFO] -- Data subjects' information is imported.")
    return dss

def set_data_types(data_types=["userAcceleration"]):
    return [[t+".x",t+".y",t+".z"] if t != "attitude" else [t+".roll", t+".pitch", t+".yaw"] for t in data_types]

def discover_dataset_structure(base_dir):
    activity_structure = {}
    sub_dirs = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    for sub_dir in sub_dirs:
        activity, trial = sub_dir.split("_")
        trial = int(trial)
        activity_structure.setdefault(activity, []).append(trial)
    for key in activity_structure:
        activity_structure[key].sort()
    return activity_structure

def creat_time_series(dt_list, act_labels, trial_structure, mode="mag", labeled=True, start=0, end=None):
    num_data_cols = len(dt_list) if mode == "mag" else len(dt_list*3)
    dataset_shape = (0, num_data_cols+7) if labeled else (0, num_data_cols)
    dataset = np.zeros(dataset_shape)

    ds_list = get_ds_infos().iloc[start:end]
    print("[INFO] -- Creating Time-Series")

    for sub_id in ds_list["code"]:
        for act_id, act in enumerate(act_labels):
            for trial in trial_structure[act]:
                fname = os.path.join(base_dir, act+'_'+str(trial), 'sub_'+str(int(sub_id))+'.csv')
                if os.path.exists(fname):
                    raw_data = pd.read_csv(fname).drop(['Unnamed: 0'], axis=1)
                    vals = np.zeros((len(raw_data), num_data_cols))
                    for x_id, axes in enumerate(dt_list):
                        if mode == "mag":
                            vals[:, x_id] = (raw_data[axes]**2).sum(axis=1)**0.5
                        else:
                            vals[:, x_id*3:(x_id+1)*3] = raw_data[axes].values

                    if labeled:
                        lbls = np.array([[act_id, sub_id-1, ds_list["weight"][sub_id-1], ds_list["height"][sub_id-1], ds_list["age"][sub_id-1], ds_list["gender"][sub_id-1], trial]]*len(raw_data))
                        vals = np.concatenate((vals, lbls), axis=1)
                    dataset = np.append(dataset, vals, axis=0)
                    del raw_data, vals
                    gc.collect()
                else:
                    print(f"File {fname} not found!")
    return dataset


In [ ]:
# Constants
base_dir = '/content/A_DeviceMotion_data/A_DeviceMotion_data'
ACT_LABELS = ["dws", "ups", "sit", "std", "wlk", "jog"]
trial_structure = discover_dataset_structure(base_dir)

sdt = ["attitude", "userAcceleration"]
dt_list = set_data_types(sdt)
dataset = pd.DataFrame(creat_time_series(dt_list, ACT_LABELS, trial_structure, mode="raw", labeled=True))


[INFO] -- Data subjects' information is imported.
[INFO] -- Creating Time-Series


# Normalize the data (sensor readings only)


In [ ]:
feature_cols = dataset.columns[:-7]
dataset[feature_cols] = (dataset[feature_cols] - dataset[feature_cols].mean()) / dataset[feature_cols].std()


# Define the necessary variables


In [ ]:
window_size = 100
stride = 50
num_features = len(feature_cols)

In [ ]:
# Determine the total number of windows
n_windows = (len(dataset) - window_size) // stride + 1

In [ ]:
# Preallocate memory
X = np.empty((n_windows, window_size, num_features), dtype=np.float32)  # Use float32 for less memory usage
y = np.empty(n_windows, dtype=np.int32)  # Assuming labels are integers

for i, start in enumerate(range(0, len(dataset) - window_size, stride)):
    X[i] = dataset[feature_cols].values[start: start + window_size]
    y[i] = dataset[6].values[start + window_size // 2]

y = to_categorical(y)


# Spliting the Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Model
model = Sequential()
model.add(Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(window_size, num_features)))
model.add(MaxPooling1D(pool_size=2))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(100, activation='relu'))
model.add(Dense(len(ACT_LABELS), activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# Training The Model

In [ ]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1)


Epoch 1/10
636/636 [==============================] - 21s 31ms/step - loss: 0.3473 - accuracy: 0.8786 - val_loss: 0.1487 - val_accuracy: 0.9540
Epoch 2/10
636/636 [==============================] - 15s 24ms/step - loss: 0.1414 - accuracy: 0.9575 - val_loss: 0.1087 - val_accuracy: 0.9655
Epoch 3/10
636/636 [==============================] - 17s 27ms/step - loss: 0.0852 - accuracy: 0.9757 - val_loss: 0.1049 - val_accuracy: 0.9659
Epoch 4/10
636/636 [==============================] - 14s 23ms/step - loss: 0.0563 - accuracy: 0.9831 - val_loss: 0.0993 - val_accuracy: 0.9717
Epoch 5/10
636/636 [==============================] - 17s 26ms/step - loss: 0.0434 - accuracy: 0.9856 - val_loss: 0.0859 - val_accuracy: 0.9770
Epoch 6/10
636/636 [==============================] - 13s 21ms/step - loss: 0.0336 - accuracy: 0.9892 - val_loss: 0.1506 - val_accuracy: 0.9584
Epoch 7/10
636/636 [==============================] - 17s 27ms/step - loss: 0.0254 - accuracy: 0.9913 - val_loss: 0.1521 - val_accuracy:

#     Convert to TensorFlow Lite


In [ ]:

# Evaluate
score = model.evaluate(X_test, y_test, verbose=1)
print("Test Loss:", score[0])
print("Test Accuracy:", score[1])

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save TFLite model
with open('activity_recognition_model.tflite', 'wb') as f:
    f.write(tflite_model)

# Download
from google.colab import files
files.download('activity_recognition_model.tflite')


177/177 [==============================] - 1s 5ms/step - loss: 0.1985 - accuracy: 0.9593
Test Loss: 0.19853903353214264
Test Accuracy: 0.9593064188957214


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!apt-get update && apt-get -qq install xxd


Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [119 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [110 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [109 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [1,085 kB]
Get:8 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease [18.1 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,254 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [1,233 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports/universe amd64 Packages [28.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [965 kB]
Hit:13 https://pp

# Conversion to .h and .cpp files

In [ ]:
# Convert to .h and .cpp
TFL_MODEL_FILENAME = 'activity_recognition_model.tflite'
TFL_H_MODEL_FILENAME = 'activity_recognition_model_data.h'
TFL_CPP_MODEL_FILENAME = 'activity_recognition_model_data.cpp'

!xxd -i {TFL_MODEL_FILENAME} > {TFL_H_MODEL_FILENAME}
REPLACE_TEXT = TFL_MODEL_FILENAME.replace('/', '_').replace('.', '_')
!sed -i 's/'{REPLACE_TEXT}'/activity_recognition_model_data/g' {TFL_H_MODEL_FILENAME}
!cp {TFL_H_MODEL_FILENAME} {TFL_CPP_MODEL_FILENAME}

# Download the generated files
from google.colab import files

# Download the .h file
files.download(TFL_H_MODEL_FILENAME)

# Download the .cpp file
files.download(TFL_CPP_MODEL_FILENAME)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Install xxd if it is not available
!apt-get -qq install xxd
# Save the file as a C source file
!xxd -i sine_model_quantized.tflite > sine_model_quantized.cc
# Print the source file
!cat sine_model_quantized.cc

xxd: sine_model_quantized.tflite: No such file or directory


In order to make the arduino work, download the arduino code from : https://github.com/Wings-hub/Gait-Pattern-Nano-33-BLE-Sense